# Intra-sample timeseries

This notebook loads intra-sample peak timeseries at the original scan-level time resolution for a matched compound/ion/isotope. The data is fetched with `load_peak_timeseries()`, which resolves compound matches, fetches per-scan data concurrently, and returns everything as a single DataFrame.

**NOTE:** Depending on the number of samples and scans per sample, loading may take some time.


In [ ]:
from mascope_sdk import MascopeClient


mascope = MascopeClient()

### Load timeseries

`load_peak_timeseries` resolves compound matches, discovers the corresponding peaks, and fetches the per-scan timeseries for each.


In [ ]:
# Load full-resolution timeseries (adjust dataset/batches/targets to your data)
# Pass a single string or a list to load multiple targets in one call.
ts = mascope.load_peak_timeseries(
    dataset="My Dataset",
    batches="My Batch",
    compound=["Succinic acid", "Lactic acid"],  # or a single: compound="CH4N2O"
    # ion="CH2O2+",       # alternative: filter by ion formula(s)
    # isotope="[13C]H2O2+",  # alternative: filter by isotope formula(s)
)
ts

### Aggregate to desired level

The raw data has one row per scan per **isotope**. Change `level` below to aggregate to ion or compound level instead.


In [ ]:
# Choose aggregation level:
#   "target_isotope_formula"  - raw per-peak (finest)
#   "target_ion_formula"      - isotopes summed per ion
#   "target_compound_formula" - ions summed per compound
level = "target_ion_formula"

agg_ts = (
    ts.groupby(["sample_item_name", "datetime_utc", level, "time"])["height"]
    .sum()
    .reset_index()
    .sort_values(["sample_item_name", "time"])
)
agg_ts

### Plot: elapsed time per sample


In [ ]:
import plotly.express as px
import plotly.io as pio


pio.templates.default = "plotly_dark"  # or "plotly_white"

fig = px.line(
    agg_ts,
    x="time",
    y="height",
    color="sample_item_name",
    symbol=level,
    title="Timeseries per sample (elapsed time)",
)
fig.update_layout(xaxis_title="Time [s]", yaxis_title="Intensity [cps]")
fig.show()

### Plot: absolute datetime


In [ ]:
fig2 = px.line(
    agg_ts,
    x="datetime_utc",
    y="height",
    color=level,
    title="Timeseries: absolute time",
)
fig2.update_layout(xaxis_title="Datetime (UTC)", yaxis_title="Intensity [cps]")
fig2.show()

### Plot: time-averaged signal


In [ ]:
import pandas as pd


time_aggregation = "10min"  # e.g. "10min", "1h"

agg_ts_avg = (
    agg_ts.groupby([level, pd.Grouper(key="datetime_utc", freq=time_aggregation)])[
        "height"
    ]
    .mean()
    .reset_index()
    .dropna()
)

fig_avg = px.line(
    agg_ts_avg,
    x="datetime_utc",
    y="height",
    color=level,
    title=f"Timeseries: {time_aggregation} average",
)
fig_avg.update_layout(
    xaxis_title="Datetime (UTC)", yaxis_title="Intensity (mean) [cps]"
)
fig_avg.show()